<a href="https://colab.research.google.com/github/Udaykiran606/Zepto-AI-ML-Capstone/blob/main/analysis/02_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# MODULE 2: PART B — PREDICTIVE MODELING & PIPELINES (02_modeling.ipynb)
# ==============================================================================

import os
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Scikit-learn preprocessing & pipeline utilities
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

# Scikit-learn estimators
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Imbalanced-learn SMOTE
from imblearn.over_sampling import SMOTE

# Scikit-learn metrics
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Configure display options and visual aesthetics
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_theme(style="whitegrid", palette="muted")

# ------------------------------------------------------------------------------
# DYNAMIC DIRECTORY & DATA LOADING SETUP
# ------------------------------------------------------------------------------
print("=" * 70)
print("READING COMMITTED CLEANED DATASET FROM DISK")
print("=" * 70)

# Determine working directory context
current_dir_name = os.path.basename(os.getcwd())

# Set output directory safely
if current_dir_name == "analytics":
    output_dir = "."
else:
    output_dir = "analytics"
    os.makedirs(output_dir, exist_ok=True)  # <-- Fixes the missing directory OSError

csv_path = os.path.join(output_dir, "titanic.csv")

# Load existing dataset or run self-healing fallback
if os.path.exists(csv_path):
    df_clean = pd.read_csv(csv_path)
    print(f"✅ Successfully loaded cleaned dataset from '{csv_path}'. Shape: {df_clean.shape}")
else:
    print(f"⚠️ '{csv_path}' not found. Executing self-healing fallback from Seaborn...")
    try:
        df_clean = sns.load_dataset("titanic")
    except Exception as e:
        raise RuntimeError(f"Failed to fetch dataset from Seaborn: {e}")

    # Fallback Cleaning steps
    if "deck" in df_clean.columns:
        df_clean.drop(columns=["deck"], inplace=True)

    df_clean["age"] = df_clean.groupby(["pclass", "sex"])["age"].transform(lambda x: x.fillna(x.median()))
    df_clean.dropna(subset=["embarked", "embark_town"], inplace=True)
    df_clean["family_size"] = df_clean["sibsp"] + df_clean["parch"] + 1

    # Save artifact safely after directory verification
    df_clean.to_csv(csv_path, index=False)
    print(f"✅ Fallback completed. Dataset saved directly to '{csv_path}'. Shape: {df_clean.shape}")

READING COMMITTED CLEANED DATASET FROM DISK
✅ Successfully loaded cleaned dataset from 'analytics/titanic.csv'. Shape: (889, 15)
